# Polymarket + Firecrawl + Mistral Workflow

This notebook walks through a small single-agent workflow step by step:

1. fetch a Polymarket market
2. collect a few public web sources with Firecrawl
3. ask Mistral Large for a simulated trade decision
4. inspect the structured JSON output

No trades are placed here — this is research and simulation only.

In [2]:
from __future__ import annotations

import json
import os
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Optional

import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import display

In [3]:
# Load local environment variables from .env if present
def load_local_env(env_path: str = '.env') -> None:
    path = Path(env_path)
    if path.exists():
        load_dotenv(path)

load_local_env()

GAMMA_BASE = 'https://gamma-api.polymarket.com'
FIRECRAWL_SEARCH_URL = 'https://api.firecrawl.dev/v1/search'
MISTRAL_API_URL = 'https://api.mistral.ai/v1/chat/completions'

print('Environment loaded.')
print('MISTRAL_API_KEY set:', bool(os.getenv('MISTRAL_API_KEY')))
print('FIRECRAWL_API_KEY set:', bool(os.getenv('FIRECRAWL_API_KEY')))

Environment loaded.
MISTRAL_API_KEY set: True
FIRECRAWL_API_KEY set: True


In [4]:
@dataclass
class MarketCandidate:
    id: str
    question: str
    slug: str | None = None
    volume24hr: float | None = None
    liquidity: float | None = None
    outcome_prices: list[float] | None = None
    clob_token_ids: list[str] | None = None

def parse_jsonish(value: Any) -> Any:
    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return value
    return value

def safe_float(value: Any) -> Optional[float]:
    try:
        if value is None or value == '':
            return None
        return float(value)
    except (TypeError, ValueError):
        return None

def parse_token_ids(raw: Any) -> list[str]:
    parsed = parse_jsonish(raw)
    if isinstance(parsed, list):
        return [str(item) for item in parsed if str(item).strip()]
    if isinstance(parsed, str):
        return [part.strip() for part in parsed.split(',') if part.strip()]
    return []

In [5]:
class PolymarketSingleAgent:
    def __init__(self) -> None:
        self.mistral_api_key = os.getenv('MISTRAL_API_KEY')
        self.firecrawl_api_key = os.getenv('FIRECRAWL_API_KEY')
        self.mistral_model = os.getenv('MISTRAL_MODEL', 'mistral-large-2512')
        self.session = requests.Session()

    def fetch_markets(self, limit: int = 10) -> list[dict[str, Any]]:
        params = {
            'limit': limit,
            'order': 'volume24hr',
            'ascending': 'false',
            'active': 'true',
            'closed': 'false',
        }
        response = self.session.get(f'{GAMMA_BASE}/markets', params=params, timeout=30)
        response.raise_for_status()
        data = response.json()
        return data if isinstance(data, list) else []

    def choose_market(self, markets: list[dict[str, Any]]) -> MarketCandidate:
        if not markets:
            raise RuntimeError('No Polymarket markets were returned.')
        market = markets[0]
        return MarketCandidate(
            id=str(market.get('id', '')),
            question=str(market.get('question', '')),
            slug=market.get('slug'),
            volume24hr=safe_float(market.get('volume24hr')),
            liquidity=safe_float(market.get('liquidity')),
            outcome_prices=parse_jsonish(market.get('outcomePrices')),
            clob_token_ids=parse_token_ids(market.get('clobTokenIds')),
        )

    def fetch_firecrawl_context(self, query: str) -> list[dict[str, Any]]:
        if not self.firecrawl_api_key:
            return []

        headers = {
            'Authorization': f'Bearer {self.firecrawl_api_key}',
            'Content-Type': 'application/json',
        }
        payload = {
            'query': query,
            'limit': 5,
            'scrapeOptions': {'formats': ['markdown'], 'onlyMainContent': True},
        }
        response = self.session.post(FIRECRAWL_SEARCH_URL, headers=headers, json=payload, timeout=60)
        response.raise_for_status()
        data = response.json()
        results = data.get('data', []) if isinstance(data, dict) else []
        return results if isinstance(results, list) else []

In [11]:
def build_prompt(market: MarketCandidate, sources: list[dict[str, Any]]) -> str:
    source_text = []
    for item in sources[:5]:
        title = item.get('title') or item.get('metadata', {}).get('title') or 'Untitled'
        url = item.get('url') or item.get('link') or ''
        snippet = item.get('markdown') or item.get('content') or item.get('description') or ''
        source_text.append(f'- {title}\n  URL: {url}\n  Snippet: {snippet[:1200]}')

    sources_block = '\n\n'.join(source_text) if source_text else 'No external sources were collected.'
    token_ids = ', '.join(market.clob_token_ids or []) or 'None found'

    return f'''
You are a Polymarket research agent.

Your job is to read one market, inspect a few public web sources, and make a simulated trading decision.
Do not place any trades. Return only valid JSON.

Market:
- question: {market.question}
- slug: {market.slug}
- volume24hr: {market.volume24hr}
- liquidity: {market.liquidity}
- outcome_prices: {market.outcome_prices}
- clob_token_ids: {token_ids}

Web context:
{sources_block}

Decision rules:
- Prefer markets where the event is well-defined, time-bounded, and researchable from public sources.
- If the evidence is weak or the market is too ambiguous, choose skip.
- If you choose a side, use 'YES' or 'NO'.
- Include a short thesis and mention the most important source(s).

Return JSON with these fields:
{{
  "decision": "buy_yes" | "buy_no" | "skip",
  "confidence": 0.0,
  "side": "YES" | "NO" | null,
  "entry_price": 0.0 | null,
  "fair_value": 0.0 | null,
  "expected_value_edge": 0.0 | null,
  "reasoning": ["..."],
  "key_sources": ["..."],
  "risk_flags": ["..."],
  "research_summary": "..."
}}
'''.strip()

In [7]:
def ask_llm(mistral_api_key: str, model: str, prompt: str) -> dict[str, Any]:
    response = requests.post(
        MISTRAL_API_URL,
        headers={
            'Authorization': f'Bearer {mistral_api_key}',
            'Content-Type': 'application/json',
        },
        json={
            'model': model,
            'messages': [{'role': 'user', 'content': prompt}],
            'response_format': {'type': 'json_object'},
            'temperature': 0.2,
        },
        timeout=90,
    )
    response.raise_for_status()
    payload = response.json()
    content = payload['choices'][0]['message']['content'] or ''
    match = re.search(r'```(?:json)?\s*([\s\S]*?)\s*```', content)
    json_text = match.group(1) if match else content
    return json.loads(json_text)

In [8]:
agent = PolymarketSingleAgent()

markets = agent.fetch_markets(limit=10)
print(f'Fetched {len(markets)} markets.')
display(pd.DataFrame(markets)[['id', 'question', 'slug', 'volume24hr', 'liquidity']].head(5))

market = agent.choose_market(markets)
print('Selected market:')
display(pd.DataFrame([market.__dict__]))

Fetched 10 markets.


,id,question,slug,volume24hr,liquidity
0,1640919,US forces enter Iran by April 30?,us-forces-enter-iran-by-april-30-899,3.626538e+07,2944842.79408
1,1455604,Will Trump talk to Xi Jinping in March?,will-trump-talk-to-xi-jinping-in-march-165,3.666496e+06,1202618.86368
2,1394299,US forces enter Iran by December 31?,us-forces-enter-iran-by-december-31-573-642-38...,3.442131e+06,732321.59374
3,1856503,Spread: Suns (-10.5),nba-phx-chi-2026-04-05-spread-away-10pt5,2.227014e+06,363190.84392
4,1706788,US x Iran ceasefire by April 7?,us-x-iran-ceasefire-by-april-7-278,2.020532e+06,188164.82932


Selected market:


,id,question,slug,volume24hr,liquidity,outcome_prices,clob_token_ids
0,1640919,US forces enter Iran by April 30?,us-forces-enter-iran-by-april-30-899,3.626538e+07,2.944843e+06,"[0.9975, 0.0025]",[291618412020622374983984964487770747035494602...


In [9]:
sources = agent.fetch_firecrawl_context(market.question)
if not sources:
    sources = agent.fetch_firecrawl_context(f'"{market.question}"')

print(f'Collected {len(sources)} Firecrawl sources.')
display(pd.DataFrame([{
    'title': s.get('title') or s.get('metadata', {}).get('title'),
    'url': s.get('url') or s.get('link'),
    'snippet': (s.get('markdown') or s.get('content') or s.get('description') or '')[:250],
} for s in sources]).head(5))

Collected 5 Firecrawl sources.


,title,url,snippet
0,Odds of US forces entering Iran by April 30 ri...,https://cryptobriefing.com/odds-of-us-forces-e...,[![Nexo](https://cryptobriefing.com/wp-content...
1,Iran WARNS U.S as Forces ENTER Iran to Hunt 'M...,https://www.youtube.com/watch?v=u6I4vnDNrMc,![Thumbnail (1920x1080)](https://i.ytimg.com/v...
2,US Ground Forces Arrive in Middle East as Iran...,https://www.military.com/benefits/2026/03/30/u...,;\n\n# US Ground Forces Arrive in Middle East ...
3,US forces' odds of entering Iran by April 30 r...,https://cryptobriefing.com/us-forces-odds-of-e...,"[![Nexo](data:image/svg+xml,%3Csvg%20xmlns='ht..."
4,"Iran War Live: Race To Find Missing Airman, Te...",https://www.newsweek.com/iran-war-live-missing...,[Iran War Live Updates\\\nRace To Find Missing...


In [12]:
if not agent.mistral_api_key:
    raise RuntimeError('MISTRAL_API_KEY is not set.')

prompt = build_prompt(market, sources)
print(prompt[:3000])

You are a Polymarket research agent.

Your job is to read one market, inspect a few public web sources, and make a simulated trading decision.
Do not place any trades. Return only valid JSON.

Market:
- question: US forces enter Iran by April 30?
- slug: us-forces-enter-iran-by-april-30-899
- volume24hr: 36265384.13846888
- liquidity: 2944842.79408
- outcome_prices: ['0.9975', '0.0025']
- clob_token_ids: 2916184120206223749839849644877707470354946028257066951797428049170871002238, 76533108781962275310651165149634079251899733930834190485860627580128626747247

Web context:
- Odds of US forces entering Iran by April 30 rise to 86% amid ...
  URL: https://cryptobriefing.com/odds-of-us-forces-entering-iran-by-april-30-rise-to-86-amid-escalating-tensions/
  Snippet: [![Nexo](https://cryptobriefing.com/wp-content/themes/cryptobriefing-lovable/nexo-mark.png)\\
Earn with Nexo](https://nexo.com/?utm_source=cryptobriefing&utm_campaign=cryptobriefing_button_earn_website)

[![Crypto Briefing](https

In [13]:
decision = ask_llm(agent.mistral_api_key, agent.mistral_model, prompt)
print(json.dumps(decision, indent=2, ensure_ascii=False))

{
  "decision": "buy_no",
  "confidence": 0.85,
  "side": "NO",
  "entry_price": 0.0025,
  "fair_value": 0.05,
  "expected_value_edge": 0.0475,
  "reasoning": [
    "The current market price for 'NO' (0.0025) is significantly undervalued compared to the available evidence.",
    "Recent reports indicate U.S. ground forces have entered Iran only in limited, high-risk rescue operations (e.g., missing F-15 pilot), not a full-scale invasion or sustained presence.",
    "The April 30 deadline is imminent, and there is no credible public evidence of preparations for a large-scale U.S. military entry into Iran.",
    "The 86% odds cited in some sources appear exaggerated and lack corroboration from official military or government statements.",
    "The most concrete reports (e.g., Oneindia News, Newsweek) describe isolated incidents, not a broader military campaign."
  ],
  "key_sources": [
    "https://www.youtube.com/watch?v=u6I4vnDNrMc",
    "https://www.newsweek.com/iran-war-live-missing-